In [ ]:
# requirements:
# pip install pandas numpy pyarrow sentence-transformers

import os
from typing import List, Tuple
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# === Simple constants you can change in one place ===
CSV_PATH   = "csv_files/locations_reviews.csv"
OUT_DIR    = "embeddings_out"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 128

TEXT_COL     = "review_text"
LOC_COL      = "location"
REVIEWID_COL = "review_text"


# 1) Load CSV(s) -> DataFrame

def load_reviews(csv_path: str) -> pd.DataFrame:
    """Load reviews from a CSV file or a directory of CSVs. Returns a DataFrame."""
    if os.path.isdir(csv_path):
        files = [os.path.join(csv_path, f) for f in os.listdir(csv_path) if f.endswith(".csv")]
        if not files:
            raise FileNotFoundError(f"No CSV files found in directory: {csv_path}")
        df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    else:
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV not found: {csv_path}")
        df = pd.read_csv(csv_path)

    # Normalize column names once
    df.columns = [c.strip().lower() for c in df.columns]
    # Basic schema check
    for col in [REVIEWID_COL, LOC_COL, TEXT_COL]:
        if col not in df.columns:
            raise KeyError(f"Missing required column '{col}'. Found: {df.columns.tolist()}")
    return df


# 2) Clean text (lightweight)

def clean_text(df: pd.DataFrame) -> pd.DataFrame:
    """Minimal, safe cleaning: cast to str, trim spaces, lowercase. Drops empty rows."""
    s = (
        df[TEXT_COL]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.lower()
    )
    df = df.assign(**{TEXT_COL: s})
    df = df[df[TEXT_COL].str.len() > 0].copy()
    # Optional: make ordering deterministic for reproducibility
    df = df.sort_values(REVIEWID_COL).reset_index(drop=True)
    return df


# 3) Model + embedding helpers

def load_model(model_name: str) -> SentenceTransformer:
    """Load SBERT model (CPU by default; will use GPU if available/env configured)."""
    return SentenceTransformer(model_name)

def embed_texts(model: SentenceTransformer, texts: List[str], batch_size: int) -> np.ndarray:
    """Embed a list of texts; returns float32 array [n, d]."""
    vecs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,  # normalize explicitly next
    )
    return vecs.astype(np.float32)

def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Row-wise L2 normalization so cosine similarity == dot product."""
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, eps)


# 4) Build embeddings + aligned meta

def build_review_embeddings(df: pd.DataFrame) -> Tuple[np.ndarray, pd.DataFrame]:
    """
    Returns:
      - embeddings: np.ndarray [n_reviews, dim] (L2-normalized)
      - meta: DataFrame with [review_id, location_id] aligned row-by-row
    """
    model = load_model(MODEL_NAME)
    emb = embed_texts(model, df[TEXT_COL].tolist(), batch_size=BATCH_SIZE)
    emb = l2_normalize(emb)
    meta = df[[REVIEWID_COL, LOC_COL]].copy()
    meta["emb_row"] = np.arange(len(meta), dtype=np.int32)
    return emb, meta


# 5) Save artifacts

def save_artifacts(embeddings: np.ndarray, meta: pd.DataFrame) -> None:
    """Save embeddings (.npy) and meta (.parquet + .csv) into OUT_DIR."""
    os.makedirs(OUT_DIR, exist_ok=True)

    # Save embeddings
    np.save(os.path.join(OUT_DIR, "review_embeddings.npy"), embeddings)

    # Save metadata
    meta.to_parquet(os.path.join(OUT_DIR, "meta.parquet"), index=False)
    meta.to_csv(os.path.join(OUT_DIR, "meta.csv"), index=False)

    print(
        f"Saved:\n"
        f"- {OUT_DIR}/review_embeddings.npy\n"
        f"- {OUT_DIR}/meta.parquet\n"
        f"- {OUT_DIR}/meta.csv"
    )



# 6) One-shot runner

def run_embed_once() -> None:
    """Load -> clean -> embed -> save. Run once to prepare review vectors."""
    df = load_reviews(CSV_PATH)
    df = clean_text(df)
    embeddings, meta = build_review_embeddings(df)
    save_artifacts(embeddings, meta)
    print(f"Done. Embedded {embeddings.shape[0]} reviews (dim={embeddings.shape[1]}).")


# Optional: allow running as a script
if __name__ == "__main__":
    run_embed_once()


Batches:   0%|          | 0/245 [00:00<?, ?it/s]

Saved:
- embeddings_out/review_embeddings.npy
- embeddings_out/meta.parquet
- embeddings_out/meta.csv
Done. Embedded 31276 reviews (dim=384).
